# RQ5 — Does a Signal Quality Gate change NeuroXVocal's accuracy?

One notebook that trains every RQ5 arm with the NeuroXVocal pipeline of
`full_pipeline_whisper_medium.ipynb` (Whisper medium, DeBERTa-v3-base, 47 acoustic features,
Wav2Vec2 embeddings, batch 8, lr 1e-5, StratifiedKFold 5 folds with SEED 42, fresh model per fold,
early stopping patience 10). Only the use of the gate differs between arms.

| Arm | Training data | Question |
|---|---|---|
| A0 | all 166 recordings | baseline |
| A1 | rejected recordings removed | does removing unusable files help? |
| A2 | rejected + flagged removed | does strict filtering help despite less data? |
| A3 | rejected removed, flagged down-weighted (loss weight 0.5) | does soft use of quality help? |
| A4 | participant-only audio (interviewer turns cut out) | does removing the interviewer help? |

**Fairness rules:** all arms use the same 5 validation folds (the gate only changes each fold's
*training* part); the independent test is always all 71 test recordings (5-fold ensemble,
threshold 0.5); every arm is compared with A0 on the same recordings (exact McNemar test,
bootstrap 95% CI).

**How to run:** edit the `# EDIT ME` lines in §1, set `SMOKE = True` for a quick check (1 epoch,
1 fold), then run all cells top to bottom. A finished fold is never retrained, so after an
interruption simply run all cells again. Final table: `results/arms_comparison.md` (§7).


## 0. Setup


In [ ]:
# Uncomment if a package is missing in this kernel:
# %pip install -r ../../requirements_cls.txt
# %pip install soundfile tabulate


In [ ]:
import os
import shutil
import re
import sys
from collections import OrderedDict

import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import BCEWithLogitsLoss
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, ConcatDataset

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import KFold, StratifiedKFold

import torchaudio
from transformers import Wav2Vec2Model, Wav2Vec2Processor, AutoTokenizer, DebertaV2Model

import librosa
import parselmouth
from scipy.signal import find_peaks

import whisper
import glob
import matplotlib.pyplot as plt

import json, time, types, random
import soundfile as sf
from scipy.stats import binomtest
from torch.utils.data import Subset, DataLoader


## 1. Configuration


In [ ]:
HERE = os.path.abspath(".")   # folder of this notebook

# ---------------------------------------------------------------- paths (EDIT ME)
DATA_ROOT = "/mnt/ssd1/ali-thesis/ADReSSo"                                    # EDIT ME
TRAIN_ROOT = os.path.join(DATA_ROOT, "diagnosis-train/diagnosis/train")
TEST_ROOT = os.path.join(DATA_ROOT, "diagnosis-test/diagnosis/test-dist")

# Pre-fitted scalers used by the notebook (../../src/inference/*.pkl relative to it)
AUDIO_FEATURES_SCALER_PATH = "/mnt/ssd1/ali-thesis/NeuroXVocal/src/inference/scaler_params_audio_features.pkl"  # EDIT ME
AUDIO_EMBEDDING_SCALER_PATH = "/mnt/ssd1/ali-thesis/NeuroXVocal/src/inference/scaler_params_audio_emb.pkl"      # EDIT ME

# Output of the Signal Quality Gate (copy sqg_out/gate_decisions.csv next to this file)
GATE_DECISIONS_CSV = os.path.join(HERE, "gate_decisions.csv")

WORK_DIR = os.path.join(HERE, "work")          # features, transcripts, participant audio
RESULTS_DIR = os.path.join(HERE, "results")   # one sub-folder per arm

TEST_LABELS_PATH = os.path.join(TEST_ROOT, "task1.csv")
TEST_LABELS_ID_COLUMN = "ID"
TEST_LABELS_DX_COLUMN = "Dx"
TEST_LABELS_AD_VALUE = "ProbableAD"

# ---------------------------------------------------------------- notebook settings (unchanged)
WHISPER_MODEL_SIZE = "medium"
TEXT_EMBEDDING_MODEL = "microsoft/deberta-v3-base"
NUM_MFCC_FEATURES = 47
NUM_EMBEDDING_FEATURES = 768
AUDIO_CHANNELS = 1
CUDA = True

BATCH_SIZE = 8
EPOCHS = 200
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4          # note: the notebook's optimizer hard-codes 1e-4 as well
NUM_FOLDS = 5
SAVE_BEST_MODEL = True
EARLY_STOPPING_PATIENCE = 10
SEED = 42

# Checkpoint selection. "val_loss" = identical to the notebook (default, keeps A0 comparable
# to your earlier results). "macro_f1" is available as a sensitivity analysis.
SELECT_BY = "val_loss"   # or "macro_f1" (sensitivity analysis; use a different RESULTS_DIR)

# ---------------------------------------------------------------- RQ5 arms
# data:        which audio the features/transcripts come from ("full" or "par" = participant only)
# drop:        gate decisions whose recordings are removed from each fold's TRAINING part
# weights:     per-decision loss weight in training (A3); validation and test are never weighted
ARMS = {
    "A0": {"desc": "Baseline: all training recordings",                 "data": "full", "drop": [],                  "weights": None},
    "A1": {"desc": "Gate: rejected recordings removed from training",   "data": "full", "drop": ["reject"],          "weights": None},
    "A2": {"desc": "Gate: rejected + flagged removed from training",    "data": "full", "drop": ["reject", "flag"],  "weights": None},
    "A3": {"desc": "Gate as loss weight (accept 1.0, flag 0.5)",       "data": "full", "drop": ["reject"],          "weights": {"accept": 1.0, "flag": 0.5}},
    "A4": {"desc": "Participant-only speech (INV removed), all recordings", "data": "par", "drop": [],              "weights": None},
}

# ---------------------------------------------------------------- what to run
ARMS_TO_RUN = ["A0", "A1", "A2", "A3", "A4"]   # EDIT ME to run only some arms
SMOKE = False   # True = 1 epoch, 1 fold per arm into results_smoke/ (pipeline check)
if SMOKE:
    RESULTS_DIR = os.path.join(HERE, "results_smoke")
os.makedirs(WORK_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

for p in (TRAIN_ROOT, TEST_ROOT, AUDIO_FEATURES_SCALER_PATH, AUDIO_EMBEDDING_SCALER_PATH, GATE_DECISIONS_CSV):
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
# the functions below read settings through C.<NAME>
C = types.SimpleNamespace(**{k: v for k, v in globals().items() if k.isupper()})


## 2. NeuroXVocal functions (copied unchanged from `full_pipeline_whisper_medium.ipynb`)


### Notebook §2.1 Transcription


In [ ]:
def transcribe_audio_files(data_path, model_size="base", output_base=None):
    model = whisper.load_model(model_size)

    output_base = output_base or os.path.join(os.path.dirname(data_path), 'extracted_data')

    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.lower().endswith('.wav'):
                audio_path = os.path.join(root, file)
                relative_path = os.path.relpath(root, data_path)
                output_dir = os.path.join(output_base, relative_path)
                os.makedirs(output_dir, exist_ok=True)
                output_file = os.path.join(output_dir, os.path.splitext(file)[0] + '.txt')
                print(f"Transcribing {audio_path}...")
                result = model.transcribe(audio_path)
                text = result["text"]
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(text)
                print(f"Saved transcription to {output_file}")


### Notebook §2.2 Acoustic features


In [ ]:
def extract_features(audio_path, sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1):
    y, sr = librosa.load(audio_path, sr=sr)
    duration = librosa.get_duration(y=y, sr=sr)
    energy = np.array([
        sum(abs(y[i:i+frame_length]**2))
        for i in range(0, len(y), hop_length)
    ])
    threshold = np.percentile(energy, silence_threshold * 100)
    silence_frames = energy < threshold
    speech_frames = ~silence_frames
    total_speech_frames = np.sum(speech_frames)
    total_speech_time = total_speech_frames * hop_length / sr
    silence_durations = []
    current_silence = 0
    for is_silence in silence_frames:
        if is_silence:
            current_silence += 1
        elif current_silence > 0:
            silence_durations.append(current_silence * hop_length / sr)
            current_silence = 0
    num_pauses = len(silence_durations)
    total_pause_duration = sum(silence_durations)
    avg_pause_duration = np.mean(silence_durations) if silence_durations else 0
    max_pause_duration = max(silence_durations) if silence_durations else 0
    pause_duration_std = np.std(silence_durations) if silence_durations else 0
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    pitches = pitches.flatten()
    magnitudes = magnitudes.flatten()
    pitches = pitches[magnitudes > np.median(magnitudes)]
    pitch_mean = np.mean(pitches) if len(pitches) > 0 else 0
    pitch_std = np.std(pitches) if len(pitches) > 0 else 0
    pitch_range = np.ptp(pitches) if len(pitches) > 0 else 0
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    intensity = librosa.amplitude_to_db(S, ref=np.max)
    intensity_mean = np.mean(intensity)
    intensity_std = np.std(intensity)
    intensity_range = np.ptp(intensity)

    spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)[0]
    spectral_centroid_mean = np.mean(spectral_centroids)
    spectral_centroid_std = np.std(spectral_centroids)

    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)[0]
    zcr_mean = np.mean(zcr)
    zcr_std = np.std(zcr)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=frame_length, hop_length=hop_length)
    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)

    articulation_rate = total_speech_time / (duration - total_pause_duration) if (duration - total_pause_duration) > 0 else 0

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    peaks = find_peaks(onset_env, height=np.mean(onset_env))[0]
    estimated_syllables = len(peaks)
    speaking_rate = estimated_syllables / total_speech_time if total_speech_time > 0 else 0

    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch(time_step=hop_length/sr)
    point_process = parselmouth.praat.call([snd, pitch], "To PointProcess (cc)")
    local_jitter = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3)
    local_shimmer = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3, 1.6)
    harmonicity = snd.to_harmonicity_cc(time_step=hop_length/sr)
    hnr_values = harmonicity.values
    hnr_mean = np.mean(hnr_values[hnr_values != 0]) if np.any(hnr_values != 0) else 0

    formant = snd.to_formant_burg(time_step=hop_length/sr)
    f1 = []
    f2 = []
    f3 = []
    for t in np.arange(0, snd.duration, hop_length / sr):
        try:
            f1.append(formant.get_value_at_time(1, t))
            f2.append(formant.get_value_at_time(2, t))
            f3.append(formant.get_value_at_time(3, t))
        except:
            continue
    f1 = np.array(f1)
    f2 = np.array(f2)
    f3 = np.array(f3)
    formant_1_mean = np.mean(f1) if len(f1) > 0 else 0
    formant_1_std = np.std(f1) if len(f1) > 0 else 0
    formant_2_mean = np.mean(f2) if len(f2) > 0 else 0
    formant_2_std = np.std(f2) if len(f2) > 0 else 0
    formant_3_mean = np.mean(f3) if len(f3) > 0 else 0
    formant_3_std = np.std(f3) if len(f3) > 0 else 0

    features = {
        'duration': duration,
        'total_speech_time': total_speech_time,
        'speech_pause_ratio': total_speech_time / duration if duration > 0 else 0,
        'num_pauses': num_pauses,
        'total_pause_duration': total_pause_duration,
        'avg_pause_duration': avg_pause_duration,
        'max_pause_duration': max_pause_duration,
        'pause_duration_std': pause_duration_std,
        'pitch_mean': pitch_mean,
        'pitch_std': pitch_std,
        'pitch_range': pitch_range,
        'intensity_mean': intensity_mean,
        'intensity_std': intensity_std,
        'intensity_range': intensity_range,
        'articulation_rate': articulation_rate,
        'speaking_rate': speaking_rate,
        'spectral_centroid_mean': spectral_centroid_mean,
        'spectral_centroid_std': spectral_centroid_std,
        'zcr_mean': zcr_mean,
        'zcr_std': zcr_std,
        'jitter_local': local_jitter,
        'shimmer_local': local_shimmer,
        'hnr_mean': hnr_mean,
        'formant_1_mean': formant_1_mean,
        'formant_1_std': formant_1_std,
        'formant_2_mean': formant_2_mean,
        'formant_2_std': formant_2_std,
        'formant_3_mean': formant_3_mean,
        'formant_3_std': formant_3_std,
    }

    for i, (mean, std) in enumerate(zip(mfccs_mean, mfccs_std)):
        features[f'mfcc_{i+1}_mean'] = mean
        features[f'mfcc_{i+1}_std'] = std

    return features

def process_audio_feature_files(data_path, output_csv, sr, frame_length, hop_length, silence_threshold):
    all_features = []
    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.lower().endswith('.wav'):

                audio_path = os.path.join(root, file)
                patient_id = os.path.splitext(file)[0]
                print(f"Processing {audio_path}...")
                try:
                    features = extract_features(
                        audio_path,
                        sr=sr,
                        frame_length=frame_length,
                        hop_length=hop_length,
                        silence_threshold=silence_threshold
                    )
                    features['patient_id'] = patient_id
                    features['class'] = os.path.basename(root)
                    all_features.append(features)
                except Exception as e:
                    print(f"Error processing {audio_path}: {e}")
    df = pd.DataFrame(all_features)
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Features saved to {output_csv}")


### Notebook §2.3 Wav2Vec2 embeddings


In [ ]:
def extract_embeddings(audio_path, model, processor, device):
    speech_array, sampling_rate = torchaudio.load(audio_path)
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0)
        print(f"Converted stereo to mono for {audio_path}")
    else:
        speech_array = speech_array.squeeze(0)

    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
        speech_array = resampler(speech_array)
        sampling_rate = 16000

    speech = speech_array.numpy()

    inputs = processor(speech, sampling_rate=sampling_rate, return_tensors="pt", padding=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state
    embedding = embeddings.mean(dim=1).squeeze().cpu().numpy()

    return embedding

def process_audio_embedding_files(data_path, output_csv, model_name='facebook/wav2vec2-base-960h'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name)
    model.to(device)
    model.eval()

    all_embeddings = []
    for root, dirs, files in os.walk(data_path):
        for file in tqdm(files, desc=f"Processing {root}"):
            if file.lower().endswith('.wav'):
                audio_path = os.path.join(root, file)
                patient_id = os.path.splitext(file)[0]
                try:
                    embedding = extract_embeddings(audio_path, model, processor, device)
                    all_embeddings.append({'patient_id': patient_id, 'embedding': embedding})
                except Exception as e:
                    print(f"Error processing {audio_path}: {e}")
                    continue  # Skip to the next file

    if len(all_embeddings) == 0:
        print("No embeddings were extracted. Please check for errors.")
        return

    df = pd.DataFrame(all_embeddings)
    # Expand the 'embedding' column into separate columns
    embedding_cols = pd.DataFrame(df['embedding'].tolist())
    embedding_cols['patient_id'] = df['patient_id']
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    embedding_cols.to_csv(output_csv, index=False)
    print(f"Embeddings saved to {output_csv}")


### Notebook §3.1 Text cleaning


In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = ' '.join(text.split())
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)

    return text

def process_text_files(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for filename in os.listdir(input_dir):
        if filename.lower().endswith('.txt'):
            input_file = os.path.join(input_dir, filename)
            output_file = os.path.join(output_dir, filename)
            with open(input_file, 'r', encoding='utf-8') as f:
                text = f.read()
            processed_text = preprocess_text(text)
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(processed_text)

            print(f"Processed {filename}")


### Notebook §3.2 Audio feature scaling


In [ ]:
def preprocess_csv(input_path, output_path, scaler_path):
    df = pd.read_csv(input_path)
    df = df.drop(['jitter_local', 'shimmer_local', 'formant_1_mean',
                  'formant_1_std', 'formant_2_mean', 'formant_2_std',
                  'formant_3_mean', 'formant_3_std', 'class'], axis=1)

    patient_ids = df['patient_id']
    df = df.drop(['patient_id'], axis=1)
    df = df.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = joblib.load(scaler_path)
    scaled_features = scaler.transform(df)
    df_scaled = pd.DataFrame(scaled_features, columns=df.columns)
    df_scaled['patient_id'] = patient_ids
    output_file = os.path.join(output_path, os.path.basename(input_path))
    df_scaled.to_csv(output_file, index=False)
    print(f"Processed file saved at: {output_file}")


### Notebook §3.3 Audio embedding scaling


In [ ]:
def process_csv(input_csv, scaler_path, output_csv):
    df = pd.read_csv(input_csv)
    patient_ids = df['patient_id']
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()), axis=0)
    scaler = joblib.load(scaler_path)
    standardized_features = scaler.transform(features)
    standardized_df = pd.DataFrame(standardized_features, columns=features.columns)
    standardized_df['patient_id'] = patient_ids
    standardized_df.to_csv(output_csv, index=False)
    print(f"Processed CSV saved to '{output_csv}'")


### Notebook §4.1 Dataset


In [ ]:
class DementiaDataset(Dataset):
    def __init__(
        self,
        audio_csv_path,
        embedding_csv_path,
        text_dir,
        label_value=None,
        tokenizer_model=TEXT_EMBEDDING_MODEL,
        max_length=512
    ):
        self.audio_data = pd.read_csv(audio_csv_path)
        self.embedding_data = pd.read_csv(embedding_csv_path)
        self.text_dir = text_dir
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.max_length = max_length
        self.data = pd.merge(
            self.audio_data,
            self.embedding_data,
            on='patient_id',
            suffixes=('_audio', '_embedding')
        )

        self.label_value = label_value
        self.data['label'] = self.label_value

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_row = self.data.iloc[idx]
        patient_id = data_row['patient_id']
        text_file_path = os.path.join(self.text_dir, f'{patient_id}.txt')

        with open(text_file_path, 'r') as file:
            text = file.read()

        text_tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )


        audio_feature_columns = [
            col for col in self.audio_data.columns if col != 'patient_id'
        ]
        embedding_feature_columns = [
            col for col in self.embedding_data.columns if col != 'patient_id'
        ]
        audio_features = data_row[audio_feature_columns].values.astype(float)
        embedding_features = data_row[embedding_feature_columns].values.astype(float)
        label = torch.tensor(data_row['label'], dtype=torch.float32)

        audio_tensor = torch.tensor(audio_features, dtype=torch.float32)
        embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32)

        return text_tokens, audio_tensor, embedding_tensor, label

def create_full_dataset(
    ad_text_dir,
    cn_text_dir,
    ad_csv,
    cn_csv,
    ad_embedding_csv,
    cn_embedding_csv
):
    ad_dataset = DementiaDataset(
        ad_csv,
        ad_embedding_csv,
        ad_text_dir,
        label_value=1
    )
    cn_dataset = DementiaDataset(
        cn_csv,
        cn_embedding_csv,
        cn_text_dir,
        label_value=0
    )
    full_dataset = ConcatDataset([ad_dataset, cn_dataset])
    return full_dataset


### Notebook §4.2 Model


In [ ]:
class NeuroXVocal(nn.Module):
    def __init__(
        self,
        num_audio_features,
        num_embedding_features,
        text_embedding_model,
        hidden_size=768
    ):
        super(NeuroXVocal, self).__init__()

        self.text_model = DebertaV2Model.from_pretrained(text_embedding_model)
        self.hidden_size = self.text_model.config.hidden_size
        self.audio_fc = nn.Sequential(
            nn.Linear(num_audio_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        self.embedding_fc = nn.Sequential(
            nn.Linear(num_embedding_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=8,
            dropout=0.35,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.45),
            nn.Linear(self.hidden_size // 2, 1)
        )

    def forward(self, text_input, audio_input, embedding_input):
        input_ids = text_input['input_ids'].squeeze(1)
        attention_mask = text_input['attention_mask'].squeeze(1)
        text_outputs = self.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_embeddings = text_outputs.last_hidden_state
        audio_embeddings = self.audio_fc(audio_input)
        audio_embeddings = audio_embeddings.unsqueeze(1)
        embedding_embeddings = self.embedding_fc(embedding_input)
        embedding_embeddings = embedding_embeddings.unsqueeze(1)
        combined_embeddings = torch.cat(
            (audio_embeddings, embedding_embeddings, text_embeddings),
            dim=1
        )
        combined_embeddings = combined_embeddings.permute(1, 0, 2)
        transformer_output = self.transformer_encoder(combined_embeddings)
        pooled_output = transformer_output[0]
        logits = self.classifier(pooled_output).squeeze(-1)

        return logits

    def reset_parameters(self):
        def reset_layer_parameters(layer):
            if hasattr(layer, 'reset_parameters'):
                layer.reset_parameters()
        self.audio_fc.apply(reset_layer_parameters)
        self.embedding_fc.apply(reset_layer_parameters)
        self.transformer_encoder.apply(reset_layer_parameters)
        self.classifier.apply(reset_layer_parameters)


### Notebook §6.3 Test dataset


In [ ]:
class TestPatientDataset(Dataset):
    """Like DementiaDataset, but for the independent test set: each patient's label is
    looked up from a separate ground-truth file (test-dist/audio/ has no per-class
    subfolders, unlike the training data)."""
    def __init__(
        self,
        audio_csv_path,
        embedding_csv_path,
        text_dir,
        labels_df,
        id_column,
        dx_column,
        ad_value,
        tokenizer_model=TEXT_EMBEDDING_MODEL,
        max_length=512
    ):
        self.audio_data = pd.read_csv(audio_csv_path)
        self.embedding_data = pd.read_csv(embedding_csv_path)
        self.text_dir = text_dir
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.max_length = max_length

        self.data = pd.merge(
            self.audio_data,
            self.embedding_data,
            on='patient_id',
            suffixes=('_audio', '_embedding')
        )

        label_lookup = dict(zip(labels_df[id_column], labels_df[dx_column]))
        missing = self.data[~self.data['patient_id'].isin(label_lookup)]
        if len(missing) > 0:
            print(f"Warning: {len(missing)} test patients have no matching ground-truth "
                  f"label and will be excluded: {missing['patient_id'].tolist()}")
            self.data = self.data[self.data['patient_id'].isin(label_lookup)].reset_index(drop=True)

        self.data['label'] = self.data['patient_id'].map(
            lambda pid: 1 if label_lookup[pid] == ad_value else 0
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_row = self.data.iloc[idx]
        patient_id = data_row['patient_id']
        text_file_path = os.path.join(self.text_dir, f'{patient_id}.txt')

        with open(text_file_path, 'r') as file:
            text = file.read()

        text_tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        audio_feature_columns = [
            col for col in self.audio_data.columns if col != 'patient_id'
        ]
        embedding_feature_columns = [
            col for col in self.embedding_data.columns if col != 'patient_id'
        ]
        audio_features = data_row[audio_feature_columns].values.astype(float)
        embedding_features = data_row[embedding_feature_columns].values.astype(float)
        label = torch.tensor(data_row['label'], dtype=torch.float32)

        audio_tensor = torch.tensor(audio_features, dtype=torch.float32)
        embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32)

        return text_tokens, audio_tensor, embedding_tensor, label, patient_id


## 3. Participant-only audio (input of arm A4)

PAR turns from the official segmentation are cut out and concatenated with 10 ms raised-cosine
fades (no clicks / edge ringing). Files without PAR turns (5 controls) keep everything except
the INV turns. Original sample rate and sample format are kept.


In [ ]:
#!/usr/bin/env python3
"""
Arm A4 input: participant-only audio from the official ADReSSo segmentation files.

For every recording the PAR turns are cut out and concatenated with 10 ms raised-cosine
fades at each boundary (avoids clicks / edge ringing at cut points). Files whose
segmentation has no PAR turns (5 control recordings) fall back to "everything except the
INV turns". Output keeps the original sample rate and sample format (PCM_16 / FLOAT), so
the only change relative to arm A0 is the removal of interviewer speech.

Writes: work/audio_par/train/{ad,cn}/*.wav , work/audio_par/test/*.wav , work/audio_par/log.csv
"""
FADE_S = 0.010


def intervals(seg_csv, speaker):
    df = pd.read_csv(seg_csv)
    df = df[df["speaker"].astype(str).str.upper().str.startswith(speaker)]
    return [(b / 1000.0, e / 1000.0) for b, e in zip(df["begin"], df["end"]) if e > b]


def merge(iv):
    iv = sorted(iv); out = []
    for b, e in iv:
        if out and b <= out[-1][1]:
            out[-1] = (out[-1][0], max(out[-1][1], e))
        else:
            out.append((b, e))
    return out


def complement(iv, dur):
    out, t = [], 0.0
    for b, e in merge(iv):
        if b > t:
            out.append((t, b))
        t = max(t, e)
    if t < dur:
        out.append((t, dur))
    return out


def cut(x, sr, iv):
    n = int(FADE_S * sr); ramp = 0.5 - 0.5 * np.cos(np.linspace(0, np.pi, n)) if n > 1 else None
    pieces = []
    for b, e in merge(iv):
        s = x[int(b * sr):int(e * sr)].copy()
        if len(s) < 2 * max(n, 1):
            continue
        if ramp is not None:
            s[:n] *= ramp[:, None] if s.ndim == 2 else ramp
            s[-n:] *= ramp[::-1, None] if s.ndim == 2 else ramp[::-1]
        pieces.append(s)
    return np.concatenate(pieces) if pieces else x[:0]


def process(wav, seg_csv, out_wav, log):
    info = sf.info(wav)
    x, sr = sf.read(wav, dtype="float32")
    dur = len(x) / sr
    par = intervals(seg_csv, "PAR") if os.path.exists(seg_csv) else []
    src = "segmentation_PAR"
    if not par:
        inv = intervals(seg_csv, "INV") if os.path.exists(seg_csv) else []
        par, src = complement(inv, dur), "complement_of_INV"
    y = cut(x, sr, par)
    if len(y) < sr:  # < 1 s left: keep the original so every recording has an input
        y, src = x, src + "_TOO_SHORT_kept_full"
    os.makedirs(os.path.dirname(out_wav), exist_ok=True)
    sf.write(out_wav, y, sr, subtype=info.subtype)
    log.append({"file": os.path.basename(wav), "source": src, "orig_s": round(dur, 2),
                "par_s": round(len(y) / sr, 2), "subtype": info.subtype})


def make_participant_audio():
    log = []
    for cls in ("ad", "cn"):
        for wav in sorted(glob.glob(os.path.join(TRAIN_ROOT, "audio", cls, "*.wav"))):
            stem = os.path.splitext(os.path.basename(wav))[0]
            process(wav, os.path.join(TRAIN_ROOT, "segmentation", cls, stem + ".csv"),
                    os.path.join(WORK_DIR, "audio_par", "train", cls, stem + ".wav"), log)
    for wav in sorted(glob.glob(os.path.join(TEST_ROOT, "audio", "*.wav"))):
        stem = os.path.splitext(os.path.basename(wav))[0]
        process(wav, os.path.join(TEST_ROOT, "segmentation", stem + ".csv"),
                os.path.join(WORK_DIR, "audio_par", "test", stem + ".wav"), log)
    df = pd.DataFrame(log)
    df.to_csv(os.path.join(WORK_DIR, "audio_par", "log.csv"), index=False)
    print(df["source"].value_counts().to_string())
    print(f"Participant-only audio written for {len(df)} recordings "
          f"(mean kept {df.par_s.sum() / df.orig_s.sum():.1%} of the original duration)")


## 4. Feature preparation (once per data variant; existing outputs are skipped)


In [ ]:
def paths(variant):
    base = os.path.join(WORK_DIR, "features", variant)
    if variant == "full":
        wav = {"ad": os.path.join(TRAIN_ROOT, "audio", "ad"), "cn": os.path.join(TRAIN_ROOT, "audio", "cn"),
               "test": os.path.join(TEST_ROOT, "audio")}
    else:
        wav = {k: os.path.join(WORK_DIR, "audio_par", "train" if k != "test" else "test", *([k] if k != "test" else []))
               for k in ("ad", "cn", "test")}
    p = {}
    for k in ("ad", "cn", "test"):
        p[k] = {
            "wav": wav[k],
            "transcripts": os.path.join(base, "transcripts", k),
            "text": os.path.join(base, "text", k),
            "feat_raw": os.path.join(base, "raw", f"audio_features_{k}_raw.csv"),
            "emb_raw": os.path.join(base, "raw", f"audio_embeddings_{k}_raw.csv"),
            "feat_dir": os.path.join(base, "processed", k),
            "feat": os.path.join(base, "processed", k, f"audio_features_{k}_raw.csv"),
            "emb": os.path.join(base, "processed", k, f"audio_embeddings_{k}.csv"),
        }
    return p


def prepare_features(variant):
    a = types.SimpleNamespace(variant=variant)
    for f in (AUDIO_FEATURES_SCALER_PATH, AUDIO_EMBEDDING_SCALER_PATH):
        if not os.path.exists(f):
            raise SystemExit(f"Scaler not found: {f}\nSet the path in config.py")

    P = paths(a.variant)
    for k, p in P.items():
        if not os.path.isdir(p["wav"]):
            raise SystemExit(f"Audio folder missing: {p['wav']}" + ("  (run make_participant_audio.py)" if a.variant == "par" else ""))
        n_wav = len([f for f in os.listdir(p["wav"]) if f.lower().endswith(".wav")])
        print(f"\n=== {a.variant} / {k}: {n_wav} wav files ===")

        n_txt = len(os.listdir(p["transcripts"])) if os.path.isdir(p["transcripts"]) else 0
        if n_txt < n_wav:
            transcribe_audio_files(p["wav"], model_size=WHISPER_MODEL_SIZE, output_base=p["transcripts"])
        else:
            print("transcripts exist - skipped")
        if not os.path.exists(p["feat_raw"]):
            process_audio_feature_files(p["wav"], p["feat_raw"], sr=22050, frame_length=2048,
                                           hop_length=512, silence_threshold=0.1)
        if not os.path.exists(p["emb_raw"]):
            process_audio_embedding_files(p["wav"], p["emb_raw"])

        os.makedirs(p["text"], exist_ok=True); os.makedirs(p["feat_dir"], exist_ok=True)
        process_text_files(p["transcripts"], p["text"])
        preprocess_csv(p["feat_raw"], p["feat_dir"], AUDIO_FEATURES_SCALER_PATH)   # same fitted scalers as the notebook
        process_csv(p["emb_raw"], AUDIO_EMBEDDING_SCALER_PATH, p["emb"])
    print(f"\nDone: work/features/{a.variant}/ is ready for training.")


## 5. Training and test evaluation of one arm


In [ ]:
def set_seed(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def feature_paths(variant):
    base = os.path.join(C.WORK_DIR, "features", variant)
    fp = lambda k: {"text": os.path.join(base, "text", k),
                    "feat": os.path.join(base, "processed", k, f"audio_features_{k}_raw.csv"),
                    "emb": os.path.join(base, "processed", k, f"audio_embeddings_{k}.csv")}
    P = {k: fp(k) for k in ("ad", "cn", "test")}
    missing = [v for d in P.values() for v in d.values() if not os.path.exists(v)]
    if missing:
        raise SystemExit("Missing prepared features (run prepare_features.py --variant %s):\n  %s"
                 % (variant, "\n  ".join(missing)))
    return P


def metrics(y, prob):
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
    y = np.asarray(y).astype(int); pred = (np.asarray(prob) > 0.5).astype(int)
    out = {"n": int(len(y)), "accuracy": accuracy_score(y, pred),
           "f1_macro": f1_score(y, pred, average="macro", zero_division=0),
           "precision_ad": precision_score(y, pred, zero_division=0),
           "recall_ad": recall_score(y, pred, zero_division=0),
           "f1_ad": f1_score(y, pred, zero_division=0)}
    try:
        out["auc"] = roc_auc_score(y, prob)
    except ValueError:
        out["auc"] = float("nan")
    return out


def run_arm(arm_name, epochs=None, folds_only=0):
    a = types.SimpleNamespace(arm=arm_name, epochs=epochs or C.EPOCHS, folds_only=folds_only)
    arm = C.ARMS[a.arm]
    out = os.path.join(C.RESULTS_DIR, a.arm); os.makedirs(out, exist_ok=True)


    set_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() and C.CUDA else "cpu")
    print(f"Arm {a.arm}: {arm['desc']}  |  device={device}  |  select_by={C.SELECT_BY}")
    if device.type != "cuda":
        print("WARNING: training on CPU will be very slow")

    # ------------------------------------------------------------------ data
    P = feature_paths(arm["data"])
    full = create_full_dataset(P["ad"]["text"], P["cn"]["text"], P["ad"]["feat"], P["cn"]["feat"],
                                  P["ad"]["emb"], P["cn"]["emb"])
    ids, labels = [], []
    for ds in full.datasets:
        ids += ds.data["patient_id"].astype(str).tolist(); labels += ds.data["label"].tolist()
    ids, labels = np.array(ids), np.array(labels).astype(int)
    print(f"Training recordings: {len(ids)} (AD {labels.sum()}, CN {len(labels) - labels.sum()})")

    gate = pd.read_csv(C.GATE_DECISIONS_CSV).set_index("id")["decision"].to_dict()
    missing = [i for i in ids if i not in gate]
    if missing:
        print(f"WARNING: {len(missing)} recordings have no gate decision -> treated as 'accept'")
    decision = np.array([gate.get(i, "accept") for i in ids])
    weight = np.ones(len(ids), dtype=np.float32)
    if arm["weights"]:
        weight = np.array([arm["weights"].get(d, 1.0) for d in decision], dtype=np.float32)
    keep = ~np.isin(decision, arm["drop"])
    print("Gate decisions (train):", pd.crosstab(decision, labels).rename(columns={0: "CN", 1: "AD"}).to_dict())

    class Weighted(Dataset):
        def __len__(self): return len(full)
        def __getitem__(self, i):
            t, au, em, lab = full[i]
            return t, au, em, lab, torch.tensor(weight[i])

    wds = Weighted()

    def build_model():
        return NeuroXVocal(num_audio_features=C.NUM_MFCC_FEATURES,
                              num_embedding_features=C.NUM_EMBEDDING_FEATURES,
                              text_embedding_model=C.TEXT_EMBEDDING_MODEL)

    def run_epoch(model, loader, train, optimizer=None):
        bce = nn.BCEWithLogitsLoss(reduction="none")
        model.train(train)
        tot, n, probs, ys = 0.0, 0, [], []
        with torch.set_grad_enabled(train):
            for t, au, em, lab, w in loader:
                t = {k: v.to(device) for k, v in t.items()}
                au, em, lab, w = au.to(device), em.to(device), lab.to(device).float(), w.to(device)
                if train:
                    optimizer.zero_grad()
                logits = model(t, au, em)
                l = bce(logits, lab)
                loss = (l * w).sum() / w.sum() if train else l.mean()
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                tot += loss.item(); n += 1
                probs += torch.sigmoid(logits).detach().cpu().numpy().tolist(); ys += lab.cpu().numpy().tolist()
        return tot / max(n, 1), np.array(probs), np.array(ys)

    # ------------------------------------------------------------------ cross-validation
    skf = StratifiedKFold(n_splits=C.NUM_FOLDS, shuffle=True, random_state=C.SEED)
    fold_rows, oof, history = [], [], {}
    log = open(os.path.join(out, "training.log"), "a")
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
        if a.folds_only and fold > a.folds_only:
            break
        ck = os.path.join(out, f"model_fold{fold}_best.pth"); fj = os.path.join(out, f"fold{fold}.json")
        if os.path.exists(ck) and os.path.exists(fj):
            r = json.load(open(fj)); fold_rows.append(r["row"]); oof += r["oof"]; history[fold] = r["history"]
            print(f"Fold {fold}: already done - loaded"); continue

        tr_used = tr_idx[keep[tr_idx]]
        row = {"fold": fold, "n_train": int(len(tr_used)), "train_ad": int(labels[tr_used].sum()),
               "train_cn": int(len(tr_used) - labels[tr_used].sum()),
               "removed_ad": int(((~keep[tr_idx]) & (labels[tr_idx] == 1)).sum()),
               "removed_cn": int(((~keep[tr_idx]) & (labels[tr_idx] == 0)).sum()),
               "n_val": int(len(va_idx))}
        print(f"\nFold {fold}/{C.NUM_FOLDS}: train {row['n_train']} (AD {row['train_ad']}/CN {row['train_cn']}, "
              f"removed AD {row['removed_ad']}/CN {row['removed_cn']}), val {row['n_val']}")

        set_seed(C.SEED + fold)
        tl = DataLoader(Subset(wds, tr_used), batch_size=C.BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(C.SEED))
        vl = DataLoader(Subset(wds, va_idx), batch_size=C.BATCH_SIZE, shuffle=False)
        model = build_model()
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model.to(device)
        opt = optim.Adam(model.parameters(), lr=C.LEARNING_RATE, weight_decay=1e-4)
        sched = ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)

        best, best_ep, bad, best_probs, hist = None, 0, 0, None, {"train_loss": [], "val_loss": [], "val_f1": []}
        t0 = time.time()
        for ep in range(1, a.epochs + 1):
            trl, _, _ = run_epoch(model, tl, True, opt)
            vll, vp, vy = run_epoch(model, vl, False)
            m = metrics(vy, vp)
            hist["train_loss"].append(trl); hist["val_loss"].append(vll); hist["val_f1"].append(m["f1_macro"])
            sched.step(vll)
            score = (-vll, 0) if C.SELECT_BY == "val_loss" else (m["f1_macro"], -vll)
            improved = best is None or score > best
            if improved:
                best, best_ep, bad, best_probs = score, ep, 0, vp.copy()
                sd = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                torch.save(sd, ck)
                row.update({"best_epoch": ep, "val_loss": vll, **{f"val_{k}": v for k, v in m.items() if k != "n"}})
            else:
                bad += 1
            line = (f"[{a.arm}] fold {fold} ep {ep:3d} train_loss {trl:.4f} val_loss {vll:.4f} "
                    f"val_acc {m['accuracy']:.3f} val_f1 {m['f1_macro']:.3f}{'  *' if improved else ''}")
            print(line); log.write(line + "\n"); log.flush()
            if bad >= C.EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {ep} (best epoch {best_ep})"); break
        row["minutes"] = round((time.time() - t0) / 60, 1)
        fo = [{"patient_id": ids[i], "label": int(labels[i]), "prob": float(p), "fold": fold,
               "gate_decision": decision[i]} for i, p in zip(va_idx, best_probs)]
        json.dump({"row": row, "oof": fo, "history": hist}, open(fj, "w"))
        fold_rows.append(row); oof += fo; history[fold] = hist
        del model, opt, sched
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    log.close()

    fm = pd.DataFrame(fold_rows); fm.to_csv(os.path.join(out, "cv_fold_metrics.csv"), index=False)
    oo = pd.DataFrame(oof); oo["pred"] = (oo.prob > 0.5).astype(int); oo.to_csv(os.path.join(out, "oof_predictions.csv"), index=False)
    json.dump(history, open(os.path.join(out, "history.json"), "w"))
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, len(history), figsize=(4 * len(history), 3.2), sharey=True, squeeze=False)
        for axi, (f, h) in zip(ax[0], sorted(history.items())):
            axi.plot(h["train_loss"], label="train"); axi.plot(h["val_loss"], label="val"); axi.set_title(f"{a.arm} fold {f}")
        ax[0][0].legend(); fig.tight_layout(); fig.savefig(os.path.join(out, "loss_curves.png"), dpi=110); plt.close(fig)
    except Exception:
        pass

    # ------------------------------------------------------------------ independent test (all 71)
    test_rows = []
    cks = sorted(glob.glob(os.path.join(out, "model_fold*_best.pth")))
    if len(cks) == C.NUM_FOLDS or a.folds_only:
        labels_df = pd.read_csv(C.TEST_LABELS_PATH)
        tds = TestPatientDataset(P["test"]["feat"], P["test"]["emb"], P["test"]["text"], labels_df,
                                    C.TEST_LABELS_ID_COLUMN, C.TEST_LABELS_DX_COLUMN, C.TEST_LABELS_AD_VALUE)
        tl = DataLoader(tds, batch_size=C.BATCH_SIZE, shuffle=False)
        all_p, tid, ty = [], None, None
        for ckp in cks:
            model = build_model(); model.load_state_dict(torch.load(ckp, map_location=device)); model.to(device).eval()
            pr, ids_, ys_ = [], [], []
            with torch.no_grad():
                for t, au, em, lab, pid in tl:
                    t = {k: v.to(device) for k, v in t.items()}
                    pr += torch.sigmoid(model(t, au.to(device), em.to(device))).cpu().numpy().tolist()
                    ids_ += list(pid); ys_ += lab.numpy().astype(int).tolist()
            all_p.append(pr); tid, ty = ids_, ys_
            del model
        mp = np.mean(all_p, axis=0)
        td = pd.DataFrame({"patient_id": tid, "label": ty, "prob": mp, "pred": (mp > 0.5).astype(int)})
        td["gate_decision"] = td.patient_id.map(lambda i: gate.get(i, "accept"))
        td.to_csv(os.path.join(out, "test_predictions.csv"), index=False)
        test_all = metrics(td.label, td.prob)
        test_by = {d: metrics(g.label, g.prob) for d, g in td.groupby("gate_decision") if len(g) > 1}
    else:
        test_all, test_by = {}, {}
        print("Test evaluation skipped: not all fold checkpoints exist")

    # ------------------------------------------------------------------ summary
    cvm = {k: [float(fm[f"val_{k}"].mean()), float(fm[f"val_{k}"].std())]
           for k in ("accuracy", "f1_macro", "auc") if f"val_{k}" in fm}
    summary = {"arm": a.arm, "desc": arm["desc"], "select_by": C.SELECT_BY,
               "cv_mean_sd": cvm, "cv_pooled_oof": metrics(oo.label, oo.prob),
               "test": test_all, "test_by_gate_decision": test_by,
               "train_removed_total": {"ad": int(((~keep) & (labels == 1)).sum()), "cn": int(((~keep) & (labels == 0)).sum())}}
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=2, default=float)
    print("\n" + json.dumps({"cv_mean_sd": cvm, "test_accuracy": test_all.get("accuracy"),
                             "test_f1_macro": test_all.get("f1_macro")}, indent=2, default=float))
    print(f"Results in {out}")


## 6. Comparison of the arms with A0


In [ ]:
#!/usr/bin/env python3
"""
Compare all finished arms against the baseline A0 -> results/arms_comparison.md (+ .csv)

For each arm:
  * CV: validation accuracy / macro-F1 / AUC, mean +- SD over the 5 folds
  * pooled out-of-fold accuracy on all 166 training recordings
  * independent test (71 recordings): accuracy, macro-F1, AUC, precision / recall for AD
  * paired comparison with A0 on the SAME recordings:
      - exact McNemar test (test set and out-of-fold predictions)
      - bootstrap 95% CI of the accuracy difference (10,000 resamples)
  * test accuracy split by the gate decision of the test recording (accept / flag)
"""

def mcnemar(a_correct, b_correct):
    b = int(np.sum(a_correct & ~b_correct)); c = int(np.sum(~a_correct & b_correct))
    p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    return b, c, p


def boot_diff(a_correct, b_correct, n=10000, seed=42):
    rng = np.random.default_rng(seed); k = len(a_correct)
    idx = rng.integers(0, k, size=(n, k))
    d = b_correct[idx].mean(1) - a_correct[idx].mean(1)
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


def compare_arms():
    arms = [a for a in sorted(C.ARMS) if os.path.exists(os.path.join(C.RESULTS_DIR, a, "summary.json"))]
    if "A0" not in arms:
        raise SystemExit("A0 (baseline) has not finished yet.")
    S = {a: json.load(open(os.path.join(C.RESULTS_DIR, a, "summary.json"))) for a in arms}
    load = lambda a, f: pd.read_csv(os.path.join(C.RESULTS_DIR, a, f)).set_index("patient_id")
    base_t, base_o = load("A0", "test_predictions.csv"), load("A0", "oof_predictions.csv")

    rows = []
    for a in arms:
        s = S[a]; t, o = load(a, "test_predictions.csv"), load(a, "oof_predictions.csv")
        cv = s["cv_mean_sd"]
        r = {"arm": a, "description": s["desc"],
             "removed_train_AD/CN": f"{s['train_removed_total']['ad']}/{s['train_removed_total']['cn']}",
             "cv_acc": f"{cv['accuracy'][0]:.4f} ± {cv['accuracy'][1]:.4f}",
             "cv_f1_macro": f"{cv['f1_macro'][0]:.4f} ± {cv['f1_macro'][1]:.4f}",
             "cv_auc": f"{cv['auc'][0]:.4f} ± {cv['auc'][1]:.4f}" if "auc" in cv else "",
             "oof_acc": round(s["cv_pooled_oof"]["accuracy"], 4),
             "test_acc": round(s["test"]["accuracy"], 4), "test_f1_macro": round(s["test"]["f1_macro"], 4),
             "test_auc": round(s["test"]["auc"], 4),
             "test_precision_AD": round(s["test"]["precision_ad"], 4), "test_recall_AD": round(s["test"]["recall_ad"], 4)}
        for d in ("accept", "flag"):
            m = s["test_by_gate_decision"].get(d)
            r[f"test_acc_{d}"] = f"{m['accuracy']:.4f} (n={m['n']})" if m else ""
        if a != "A0":
            for name, ref, cur in (("test", base_t, t), ("oof", base_o, o)):
                common = ref.index.intersection(cur.index)
                ac = (ref.loc[common, "pred"] == ref.loc[common, "label"]).values
                bc = (cur.loc[common, "pred"] == cur.loc[common, "label"]).values
                b, c, p = mcnemar(ac, bc); lo, hi = boot_diff(ac, bc)
                r[f"{name}_delta_acc_vs_A0"] = round(bc.mean() - ac.mean(), 4)
                r[f"{name}_95CI"] = f"[{lo:+.4f}, {hi:+.4f}]"
                r[f"{name}_mcnemar"] = f"A0 only {b}, {a} only {c}, p={p:.4f}"
        rows.append(r)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(C.RESULTS_DIR, "arms_comparison.csv"), index=False)

    main_cols = ["arm", "description", "removed_train_AD/CN", "cv_acc", "cv_f1_macro", "test_acc", "test_f1_macro", "test_auc"]
    diff_cols = ["arm", "test_delta_acc_vs_A0", "test_95CI", "test_mcnemar", "oof_delta_acc_vs_A0", "oof_95CI", "oof_mcnemar"]
    gate_cols = ["arm", "test_acc_accept", "test_acc_flag"]
    def md(d):
        try:
            return d.to_markdown(index=False)
        except ImportError:
            return "```\n" + d.to_string(index=False) + "\n```"
    txt = ["# RQ5 - arms comparison\n",
           f"Checkpoint selection: {S['A0']['select_by']}. CV = 5-fold StratifiedKFold (SEED {C.SEED}), identical "
           "validation folds for all arms; the gate only changes each fold's training part. "
           "Test = all 71 ADReSSo test recordings, ensemble of the 5 fold models, threshold 0.5.\n",
           "## Performance\n", md(df[main_cols]),
           "\n\n## Paired difference to A0 (same recordings; exact McNemar, bootstrap 95% CI)\n",
           md(df[df.arm != "A0"][[c for c in diff_cols if c in df]]) if len(df) > 1 else "(only A0 finished)",
           "\n\n## Test accuracy by gate decision of the test recording\n", md(df[gate_cols])]
    open(os.path.join(C.RESULTS_DIR, "arms_comparison.md"), "w").write("\n".join(txt))
    print("\n".join(txt))


## 7. Run


### 7.1 Inputs (participant-only audio for A4, then features)


In [ ]:
if "A4" in ARMS_TO_RUN:
    make_participant_audio()
    prepare_features("par")
if any(a in ARMS_TO_RUN for a in ("A0", "A1", "A2", "A3")):
    prepare_features("full")


### 7.2 Train every arm (5 folds + test ensemble each)


In [ ]:
for arm in ARMS_TO_RUN:
    print("=" * 80, f"\nARM {arm}: {ARMS[arm]['desc']}\n" + "=" * 80)
    if SMOKE:
        run_arm(arm, epochs=1, folds_only=1)
    else:
        run_arm(arm)


### 7.3 Comparison table


In [ ]:
compare_arms()
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(RESULTS_DIR, "arms_comparison.md")).read()))
